# Bayesian networks with pyAgrum

**The room.** The university gym has a weights room and, on the other side of one wall, a studio where
group classes are held. The weights room has three sensors: a microphone that reports when the sound
level goes over a threshold, a CO&#8322; sensor, and an accelerometer bolted to the floor structure.

Two things can be going on in there, and neither of them is measured: a **class** is running next door,
and the weights room is **busy**. The system has to work out which, from the three sensors alone.

One sensor is what makes this interesting. **The microphone is in the weights room, but it hears both rooms.**

Today you build the network that does this reasoning, put your own numbers into it, watch it change its
mind, and then throw your numbers away and count them out of two weeks of logs instead.

In [ ]:
#@title Run this cell first. It installs pyAgrum and defines the helpers — you never need to read it.
!pip install -q pyagrum

import os, itertools, urllib.request
import pandas as pd
import pyagrum as gum
import pyagrum.lib.notebook as gnb

NODES = ["Class", "Busy", "Sound", "CO2", "Vibration"]
LOG_URL = ("https://raw.githubusercontent.com/lucregrassi/ambient-intelligence-labs"
           "/main/lab4-bayesian-networks/gym_log.csv")

if not os.path.exists("gym_log.csv"):
    urllib.request.urlretrieve(LOG_URL, "gym_log.csv")
log = pd.read_csv("gym_log.csv")

def new_network():
    """Five binary nodes, no arcs yet."""
    bn = gum.BayesNet("The weights room")
    for n in NODES:
        bn.add(gum.LabelizedVariable(n, n, ["no", "yes"]))
    return bn

def parents_of(bn, node):
    return list(bn.cpt(node).names[1:])

def _condition(key):
    """'Class=yes' or ('Class=yes', 'Busy=no') -> {'Class': 'yes', ...}"""
    keys = (key,) if isinstance(key, str) else key
    return dict(k.split("=") for k in keys)

def fill(bn, P):
    """Put your numbers into the network. Every number is P(node = yes)."""
    bn.cpt("Class").fillWith([1 - P["Class"], P["Class"]])
    bn.cpt("Busy").fillWith([1 - P["Busy"], P["Busy"]])
    for node in ("Sound", "Vibration"):
        for key, v in P[node].items():
            bn.cpt(node)[_condition(key)] = [1 - v, v]
    for key, v in P["CO2"].items():
        cond = _condition(key)
        if "Class" in parents_of(bn, "CO2"):    # optional arc: the same guess on both sides
            for c in ("no", "yes"):
                bn.cpt("CO2")[dict(cond, **{"Class": c})] = [1 - v, v]
        else:
            bn.cpt("CO2")[cond] = [1 - v, v]

def show(bn, size="5"):
    gnb.showBN(bn, size=size)

def show_cpts(bn):
    gnb.sideBySide(*[bn.cpt(n) for n in NODES], captions=NODES)

def ask(bn, evidence=None, size="8"):
    gnb.showInference(bn, evs=evidence or {}, size=size)

def belief(bn, evidence=None):
    ie = gum.LazyPropagation(bn)
    ie.setEvidence(evidence or {})
    ie.makeInference()
    return {n: round(100 * float(ie.posterior(n).toarray()[1]), 1) for n in ("Class", "Busy")}

def as_text(evidence):
    return ", ".join(f"{k}={v}" for k, v in evidence.items()) or "nothing observed"

def one_clue_at_a_time(bn, clues):
    """clues: a list of (node, value). Adds them one by one and reports the two situations."""
    rows, ev = [], {}
    rows.append({"what the system knows": as_text(ev), **belief(bn, ev)})
    for node, value in clues:
        ev = dict(ev, **{node: value})
        rows.append({"what the system knows": as_text(ev), **belief(bn, ev)})
    return pd.DataFrame(rows).set_index("what the system knows")

def learn(bn, n_cases):
    """Count the cases in the log and turn the counts into CPTs. Same structure, new numbers."""
    log.head(n_cases).to_csv("_subset.csv", index=False)
    learner = gum.BNLearner("_subset.csv", bn)
    learner.useSmoothingPrior(1)
    return learner.learnParameters(bn.dag())

def compare_cpt(bn_you, bn_log, node):
    parents = parents_of(bn_you, node)
    rows = []
    for combo in itertools.product(["no", "yes"], repeat=len(parents)):
        ev = dict(zip(parents, combo))
        row = {p: v for p, v in ev.items()}
        row[f"P({node}=yes) — you"] = round(float(bn_you.cpt(node)[ev][1]), 3)
        row[f"P({node}=yes) — the log"] = round(float(bn_log.cpt(node)[ev][1]), 3)
        rows.append(row)
    return pd.DataFrame(rows)

def compare_answers(bn_you, bn_log, evidence):
    return pd.DataFrame([{"": "your numbers", **belief(bn_you, evidence)},
                         {"": "numbers from the log", **belief(bn_log, evidence)}]).set_index("")

## 1. The structure

Before you run anything: on paper, in your group, draw the network. Five binary nodes &mdash;
`Class`, `Busy`, `Sound`, `CO2`, `Vibration` &mdash; and the arrows between them. An arrow goes from a
cause to the thing it affects, not from a sensor to a conclusion.

<details>
<summary><b>Draw it first, then open this</b></summary>

`Sound` has two parents, and that is the whole point of the lab: the microphone cannot tell which room
it heard. `CO2` has one parent, `Busy` &mdash; the studio has its own air handling. `Vibration` has one
parent, `Class`. And there is no arrow between `Class` and `Busy`: we are claiming that a class next
door does not change how busy the weights room is. That is an assumption about this building, not a
fact, and it is the first thing worth arguing about.
</details>

In [ ]:
bn = new_network()                  # five binary nodes, no arcs yet
bn.addArc("Class", "Sound")         # the class is heard through the wall
bn.addArc("Busy",  "Sound")         # a busy weights room is loud on its own
bn.addArc("Busy",  "CO2")           # the people in this room breathe
bn.addArc("Class", "Vibration")     # a jumping class shakes the floor
# bn.addArc("Class", "CO2")         # <- section 5 will tell you whether this one belongs here
show(bn)

`Sound` is a **collider**: two causes meeting at one effect. Everything surprising further down happens
at that node.

## 2. The numbers

The structure says *which* numbers the network needs. It does not say what they are. Here they are:
one for `Class`, one for `Busy`, four for `Sound` (one per combination of its two parents), two for
`CO2`, two for `Vibration`. **Ten numbers** &mdash; the full joint distribution over five binary
variables would have needed 31.

Nobody measured these. They are what a person who knows the building would say. Change the one marked
with an arrow if you disagree: it is the wall.

In [ ]:
P = {                                            # every number is P(the node = yes)
    "Class": 0.30,                               # a class runs about a third of opening hours
    "Busy":  0.40,

    "Sound": {
        ("Class=yes", "Busy=no"):  0.80,         # <- the wall: how much of the class comes through
        ("Class=yes", "Busy=yes"): 0.97,
        ("Class=no",  "Busy=yes"): 0.70,
        ("Class=no",  "Busy=no"):  0.05,         # the room is never completely silent
    },

    "CO2":       {"Busy=yes":  0.75, "Busy=no":  0.05},
    "Vibration": {"Class=yes": 0.65, "Class=no": 0.03},
}
fill(bn, P)
show_cpts(bn)

## 3. Asking it a question

The microphone goes over the threshold. Nothing else is known.

In [ ]:
ask(bn, {"Sound": "yes"})

The orange node is what you observed; the bars on the other four are what the network now believes.
Both situations went up, which is what you would expect: something made a noise.

## 4. One clue at a time

Now give it the other two sensors, one at a time, and watch the two situations move.

In [ ]:
one_clue_at_a_time(bn, [("Sound", "yes"), ("CO2", "no"), ("Vibration", "yes")])

Read the third row. The CO&#8322; sensor says nothing about the class &mdash; there is no path from
`CO2` to `Class` except through `Busy` &mdash; and yet `Class` goes from 55 to 71. The noise still has
to be explained by something, and a normal CO&#8322; reading has just made *the room is busy* a poor
explanation, so the other cause takes over.

Now read the last row. `Busy` ends at **18%, below the 40% it started from**, after three readings,
none of which measures how busy the room is. The class explains the noise on its own, so the noise
stops counting as evidence of a crowd. This is **explaining away**, and it only happens at a collider.

<details>
<summary><b>Before you run the next cell: what happens if the floor is still instead?</b></summary>

`Busy` goes back up, not down. With `Vibration = no` the class becomes unlikely, the noise is still
there and still needs an explanation, and the only explanation left is the room itself. Try it by
swapping `"yes"` for `"no"` in the cell above &mdash; and notice that you can predict the direction of
the change without doing any arithmetic. The arithmetic only tells you how far it moves.
</details>

## 5. Where the numbers could come from instead

`gym_log.csv` was generated for this lab. There is no such gym and nobody recorded anything: what is
real is the *shape* of the file. Two weeks of a gym's operating log, one row for every five minutes it
was open, from seven in the morning to ten at night. The last three columns are the sensors. The first
two &mdash; the ones that are never measured &mdash; are what you would have to reconstruct afterwards
from the booking system (which class was scheduled, and whether anyone checked in for it) and from the
turnstile counter.

That is what training data means for a context model, and the labels are the expensive part: they do
not come from the sensors. They come from other systems, exported once, and they are worth nothing at
18:42 on a Tuesday &mdash; the turnstile counts the whole gym and misses everybody who leaves without
badging out. Whether a building can be made to produce that column at all is the question that decides
whether you learn the CPTs or guess them.

In [ ]:
log.head()

In [ ]:
N_CASES = len(log)             # every slot in the log; try 50, then 200, then len(log) again
bn_log = learn(bn, N_CASES)
compare_cpt(bn, bn_log, "Sound")

The wall is the number you got most wrong: you said 0.80 of the class gets through, the log says 0.55.
The other three guesses were close enough.

Now set `N_CASES = 50` and run that cell again. The same column moves by tens of points &mdash; the
wall drops to 0.43, the CO&#8322; sensor jumps to 0.58 &mdash; because each of those numbers is being
counted off a handful of rows. **A CPT learned from few cases is not a
measurement; it is a guess with decimal places.** (The notebook adds one imaginary case of each kind
before counting, so that a combination nobody ever saw does not come out as a certainty.)

In [ ]:
compare_cpt(bn, bn_log, "CO2")

Two rows, because `CO2` has one parent. Now go back to section 1, **uncomment the
`Class -> CO2` arc**, run sections 1, 2 and 5 again, and look at this table: it will have four rows,
and the two `Busy = yes` rows will read about 0.46 and 0.49.

A difference that size, on eleven hundred rows, is noise. The log is telling you that a class next door does not
change the CO&#8322; in this room &mdash; so the arc you added was a claim the building does not
support. Structure is an argument you can lose to data.

## 6. The same question, asked twice

Loud room, normal CO&#8322;, still floor. The most ordinary evening in the gym, and the one case where
the two networks do not agree.

In [ ]:
EVIDENCE = {"Sound": "yes", "CO2": "no", "Vibration": "no"}
compare_answers(bn, bn_log, EVIDENCE)

In [ ]:
ask(bn_log, EVIDENCE)

With your numbers the system shrugs: 46 and 46, it cannot choose between the two situations. With the
numbers counted from the log it does choose &mdash; no class, the room is busy, 69%.

Same five nodes, same four arrows, same question. **The structure decides which questions the network
can answer; the numbers decide what it answers when the evidence is ambiguous** &mdash; which, in a
real building, is most evenings.

## 7. Where the four arrows came from

Not one of the four arrows was a guess. Each is a consequence of something you would write down in an
ontology of this building &mdash; and you wrote an ontology in lab 3.

| what the ontology says | what the network gets |
|---|---|
| the weights room is equipped with the microphone, and it is where people train | `Busy` &rarr; `Sound` |
| the studio hosts the class, and it is on the other side of one wall | `Class` &rarr; `Sound` |
| the weights room is equipped with the CO&#8322; sensor; the studio has its own air | `Busy` &rarr; `CO2`, and **no** arrow from `Class` |

The rule is mechanical. An activity a room is used for becomes a hidden node; a sensor the room is
equipped with becomes an observed node; there is an arrow between them when they meet in the same
room. Run it on the classroom file from lab 3 &mdash; `e2 equippedWith pir_e2`, `e2 usedFor
ami_lecture` &mdash; and it hands you the arrow `ami_lecture` &rarr; `pir_e2`. The third row is the arc you were
invited to try in section 5, and the log agreed with the ontology: separate air, no arrow.

**The ontology tells you which arrows are allowed. It cannot tell you any of the ten numbers.**

It is the same building described twice. An ontology says what it *means* for a room to be crowded
&mdash; a room with at least four people in it, counted from what somebody asserted. A network says
how *likely* it is that it is, when nobody counted and three sensors disagree.

## What to take away

1. An arrow is a claim about the building, and it is paid for in numbers: the moment `Sound` got a
   second parent it needed four of them instead of two.
2. Evidence travels backwards through a collider. A CO&#8322; reading changed what the system believed
   about a class in another room, along a path you drew yourself.
3. Counting cases is the only honest way to fill a CPT &mdash; and two weeks of logs still leaves some
   rows too thin to trust.